# Collecte des résultats de similarité

**Projet** : Gallica Images — Analyse d'illustrations des Métamorphoses d'Ovide  
**Date**   : Avril 2026

Ce notebook interroge l'API Fouille d'image BnF pour récupérer les illustrations
les plus similaires dans Gallica pour chacune des 184 gravures de Bernard Salomon (1557).

**Entrée**  : API BnF — endpoint `/api/ouvrages/{ark}/illustrations`  
**Sortie**  : `resultats/csv/salomon_segmente.csv` 

---

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, "..")
from gallica_utils import BASE_URL, ARK_SALOMON

import requests
import pandas as pd
import time
import os

os.makedirs("../../resultats/csv", exist_ok=True)
print(f"API : {BASE_URL}")
print(f"ARK Salomon : {ARK_SALOMON}")

API : https://galimages-search.bnf.fr
ARK Salomon : btv1b2200047r


## 2. Récupérer les illustrations Salomon avec leurs embeddings CLIP

In [2]:
r = requests.get(f"{BASE_URL}/api/ouvrages/{ARK_SALOMON}/illustrations", timeout=30)
illustrations = r.json()

# Garder uniquement celles qui ont un vecteur CLIP valide (768 dimensions)
valides = [
    illus for illus in illustrations
    if illus.get("metas", {}).get("content_embedding")
    and len(illus["metas"]["content_embedding"]) == 768
]

print(f"Illustrations totales       : {len(illustrations)}")
print(f"Illustrations avec vecteur  : {len(valides)}")

Illustrations totales       : 66
Illustrations avec vecteur  : 66


## 2.1 Chargement des vignettes segmentées

Pour chaque illustration Salomon, on charge la vignette segmentée par YOLO
depuis `segmentees/bois_salomon_rouille_lyon1557/` et on calcule
son embedding CLIP via `POST /api/image` — au lieu d'utiliser l'embedding
précalculé sur la page entière.

In [ ]:
import json
from pathlib import Path

DOSSIER_SEG = "../../data/editions_ovide/segmentees/bois_salomon_rouille_lyon1557"

def embedding_depuis_vignette(page):
    """
    Charge la vignette segmentée correspondant à la page Salomon
    et retourne son vecteur CLIP via POST /api/image.
    Retourne None si aucune vignette trouvée.
    """
    candidats = [
        f for f in os.listdir(DOSSIER_SEG)
        if f"_f{page:03d}_" in f and "_flip" not in f
    ]
    if not candidats:
        return None

    chemin = f"{DOSSIER_SEG}/{sorted(candidats)[0]}"
    with open(chemin, "rb") as f:
        img_bytes = f.read()

    r = requests.post(
        f"{BASE_URL}/api/image",
        files={"image": ("image.jpg", img_bytes, "image/jpeg")},
        timeout=30
    )
    if r.status_code == 200:
        return json.loads(r.text)
    return None

# Test sur la première illustration
page_test   = valides[0]["view_number"]
vecteur_test = embedding_depuis_vignette(page_test)
if vecteur_test:
    print(f"✓ Test page {page_test} — vecteur CLIP : {len(vecteur_test)} dimensions")
else:
    print(f"✗ Aucune vignette trouvée pour page {page_test}")

## 3. Recherche par similarité — boucle sur toutes les illustrations

Pour chaque illustration, on envoie son vecteur CLIP à l'API et on récupère les r résultats les plus similaires dans Gallica.

In [4]:
# Paramètre modifiable : nombre de résultats par illustration
N_RESULTATS = 20

resultats = []
total     = len(valides)

for i, illus in enumerate(valides):
    print(f"  {i+1}/{total} — page {illus['view_number']}...", end="\r")

    try:
        r = requests.post(
            f"{BASE_URL}/api/search",
            json={
                "type"        : "image",
                "image_vector": embedding_depuis_vignette(illus["view_number"]),
                "rows"        : N_RESULTATS,
                "start"       : 0
            },
            headers={"Content-Type": "application/json"},
            timeout=30
        )
        docs = r.json()["response"]["docs"]

        for doc in docs:
            resultats.append({
                "salomon_page" : illus["view_number"],
                "salomon_ark"  : illus["ark"],
                "score"        : round(doc.get("score", 0), 4),
                "titre"        : (doc.get("context_title")  or [""])[0][:80],
                "auteur"       : (doc.get("context_author") or [""])[0][:50],
                "date"         : str(doc.get("context_date", ""))[:10],
                "corpus"       : (doc.get("context_corpus") or [""])[0][:60],
                "technique"    : (doc.get("properties_technical_category") or [""])[0],
                "categorie"    : (doc.get("properties_form_function")       or [""])[0],
                "genre"        : (doc.get("properties_genre")               or [""])[0],
                "palette"      : ", ".join(doc.get("physical_palette")      or []),
                "chromatic_mode": doc.get("physical_chromatic_mode", ""),
                "link"         : doc.get("link", ""),
                "result_ark"   : doc.get("ark",  "")
            })

        time.sleep(0.1)

    except Exception as e:
        print(f"\n  Erreur page {illus['view_number']} : {e}")

print(f"\n✓ {len(resultats)} résultats collectés pour {total} illustrations")

  6/66 — page 37...
  Erreur page 37 : 'response'
  17/66 — page 47...
  Erreur page 47 : 'response'
  44/66 — page 73...
  Erreur page 73 : 'response'
  54/66 — page 83...
  Erreur page 83 : 'response'
  61/66 — page 90...
  Erreur page 90 : 'response'
  66/66 — page 95...
  Erreur page 95 : HTTPSConnectionPool(host='galimages-search.bnf.fr', port=443): Max retries exceeded with url: /api/search (Caused by ConnectTimeoutError(<HTTPSConnection(host='galimages-search.bnf.fr', port=443) at 0x75d84e27e420>, 'Connection to galimages-search.bnf.fr timed out. (connect timeout=30)'))

✓ 1200 résultats collectés pour 66 illustrations


## 4. Sauvegarde

In [5]:
df = pd.DataFrame(resultats)

chemin_csv = "../../resultats/csv/salomon_segmente.csv"
df.to_csv(chemin_csv, index=False)

print(f"✓ CSV sauvegardé : {chemin_csv}")
print(f"  Shape          : {df.shape}")
print(f"  Colonnes       : {df.columns.tolist()}")
df.head(5)

✓ CSV sauvegardé : ../../resultats/csv/salomon_segmente.csv
  Shape          : (1200, 14)
  Colonnes       : ['salomon_page', 'salomon_ark', 'score', 'titre', 'auteur', 'date', 'corpus', 'technique', 'categorie', 'genre', 'palette', 'chromatic_mode', 'link', 'result_ark']


,salomon_page,salomon_ark,score,titre,auteur,date,corpus,technique,categorie,genre,palette,chromatic_mode,link,result_ark
0,32,bfkfk34fxfr,0.9537,[Illustrations de Les Métamorphoses] / [Non id...,Ovide (0043 av. J.-C.-0017). Auteur du texte,1540,,estampe,Bande dessinée,Représentations humaines / Scènes,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk34fft6
1,32,bfkfk34fxfr,0.9441,[Illustrations de Roland furieux] / [Non ident...,"Arioste, L' (1474-1533). Auteur du texte",1544,,estampe,Bande dessinée,Représentations végétales,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk8hh9pt
2,32,bfkfk34fxfr,0.9432,"Figure del Vecchio Testamento , con versi tosc...",,1554,,estampe,Imagerie religieuse,Représentations humaines / Scènes,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk5r7bnn
3,32,bfkfk34fxfr,0.9427,Originalabdruck von Formschneider-Arbeiten... ...,,[1892 TO 1,Collections de la Bibliothèque nationale et un...,peinture,Histoire de l'art et Archéologie / Histoire de...,Représentations humaines / Scènes,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfkvw7zs
4,32,bfkfk34fxfr,0.9423,"Pub. Ovidii Nasonis Metamorphoseon libri XV, p...",Ovide (0043 av. J.-C.-0017). Auteur du texte,1576,,estampe,Illustration littéraire,Représentations humaines / Scènes,gris,monochrome,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk5xsz3b
